# 01.5 — Dataset release

Mục tiêu: kiểm tra split, leakage, manifest và gate trước khi bàn giao cho Phase 02. Câu hỏi: Phase 02 có thể dùng data mà không chạy lại Phase 01 không? Trả lời: có, khi release hoặc bundle đã khóa còn khớp manifest/checksum.

In [1]:
from pathlib import Path
import hashlib, json, subprocess, sys
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'data/processed/it_en_vi').is_dir(): ROOT = ROOT.parent
report = json.loads((ROOT / 'data/processed/it_en_vi/dataset_report.json').read_text(encoding='utf-8'))
pd.Series(report['split_counts'], name='pairs').to_frame()

,pairs
train,121547
validation,15253
it_test,15244


In [2]:
leakage = json.loads((ROOT / 'data/final_report/it_en_vi/leakage_report.json').read_text(encoding='utf-8'))
pd.DataFrame(leakage).T

,normalized_exact_pair_overlap,normalized_english_overlap
validation,0,0
it_test,0,0


In [3]:
manifest = json.loads((ROOT / 'data/processed/it_en_vi/dataset_manifest.json').read_text(encoding='utf-8'))
def sha256(path):
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    return digest
checks = [{'artifact': item['path'], 'checksum_ok': sha256(ROOT / item['path']) == item['sha256']} for item in manifest['artifacts']]
pd.DataFrame(checks)

,artifact,checksum_ok
0,data/processed/it_en_vi/train.parquet,True
1,data/processed/it_en_vi/train.jsonl,True
2,data/processed/it_en_vi/validation.parquet,True
3,data/processed/it_en_vi/validation.jsonl,True
4,data/processed/it_en_vi/it_test.parquet,True
5,data/processed/it_en_vi/it_test.jsonl,True
6,data/processed/it_en_vi/global_duplicates_remo...,True
7,data/processed/it_en_vi/dataset_report.json,True


In [4]:
gate = subprocess.run([sys.executable, str(ROOT / 'tools/validate_phase01_gate.py')], cwd=ROOT, capture_output=True, text=True)
print(gate.stdout)
if gate.returncode: raise RuntimeError(gate.stderr)

PHASE_01_GATE=PASS



Release chỉ được bàn giao khi checksum đều đúng, leakage bằng 0 và gate trả về `PHASE_01_GATE=PASS`.